# 01 · Function calling을 수동으로 구현하기

## 학습 목표
- 모델의 도구 **요청**과 Python의 도구 **실행**을 구분합니다.
- JSON Schema, `call_id`, 도구 결과 재전달을 직접 확인합니다.

## 개념
`질문 → 모델의 function_call → 허용된 Python 함수 → 외부 API → function_call_output → 최종 답변`
모델이 네트워크를 직접 호출하는 것이 아닙니다. 실행 권한은 애플리케이션이 갖습니다. 함수 이름을 `eval`로 실행하거나 모델이 만든 URL로 요청하지 않습니다.

In [ ]:
from pathlib import Path
import sys
import json
from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
from hanbit_assistant.config import Settings
from hanbit_assistant.clients import create_openai_client
settings = Settings.from_env()
# 비밀 키, 환경변수 전체, HTTP 헤더를 출력하지 않습니다.
print({"model": settings.model, "my_index": settings.search_index})


### 1. LLM 없이 도구를 먼저 시험하기
날씨는 Open-Meteo의 현재 값입니다. 환율은 Frankfurter의 최근 영업일 참고값이며 회사 정산 환율이나 은행 견적이 아닙니다. 장애가 나면 가짜 값으로 바꾸지 않고 오류를 표시합니다.

In [ ]:
from hanbit_assistant.tools import get_weather, convert_currency
print(json.dumps(await get_weather("서울"), ensure_ascii=False, indent=2))
print(json.dumps(await convert_currency(100, "USD", "KRW"), ensure_ascii=False, indent=2))


### 2. 모델에게 보여줄 도구 설명서
`description`은 언제 도구가 필요한지, `parameters`는 입력 구조를 설명합니다. `additionalProperties=False`와 필수 인수를 명시합니다.

In [ ]:
tool_schemas = [
    {"type": "function", "name": "get_weather", "description": "서울, 도쿄, 런던, 뉴욕의 현재 날씨 조회",
     "strict": True, "parameters": {"type": "object", "properties": {
         "city": {"type": "string", "description": "지원 도시 이름"}},
         "required": ["city"], "additionalProperties": False}},
    {"type": "function", "name": "convert_currency", "description": "최근 영업일 참고 환율로 금액 환산",
     "strict": True, "parameters": {"type": "object", "properties": {
         "amount": {"type": "number"}, "from_currency": {"type": "string"},
         "to_currency": {"type": "string"}},
         "required": ["amount", "from_currency", "to_currency"], "additionalProperties": False}},
]
allowed_functions = {"get_weather": get_weather, "convert_currency": convert_currency}
print([t["name"] for t in tool_schemas])


### 3. 수동 실행 루프
아래 코드는 실제 모델과 실제 API를 호출합니다. reasoning의 `encrypted_content`는 저장 없이 다음 호출을 이어가기 위한 **불투명한 암호화 데이터**입니다. 사람이 읽는 추론 과정으로 출력하지 않습니다.

In [ ]:
history = [{"role": "user", "content": "도쿄 현재 날씨와 100 USD의 KRW 참고 환산액을 알려줘."}]
async with create_openai_client(settings) as client:
    for round_number in range(5):
        result = await client.responses.create(
            model=settings.model, input=history, tools=tool_schemas,
            parallel_tool_calls=False, store=False, max_output_tokens=650,
            include=["reasoning.encrypted_content"],
        )
        calls = [item for item in result.output if item.type == "function_call"]
        if not calls:
            print("최종 답변:", result.output_text)
            break
        for item in result.output:
            if item.type in {"reasoning", "function_call"}:
                fields = ({"type", "id", "summary", "encrypted_content", "status"}
                          if item.type == "reasoning"
                          else {"type", "id", "call_id", "name", "arguments", "status"})
                history.append({k: v for k, v in item.model_dump(exclude_none=True).items() if k in fields})
        for call in calls:
            if call.name not in allowed_functions:
                raise ValueError("허용되지 않은 도구 요청")
            arguments = json.loads(call.arguments)
            print("도구 요청:", call.name, arguments)
            tool_result = await allowed_functions[call.name](**arguments)
            print("도구 결과:", json.dumps(tool_result, ensure_ascii=False))
            history.append({"type": "function_call_output", "call_id": call.call_id,
                            "output": json.dumps(tool_result, ensure_ascii=False)})
    else:
        raise RuntimeError("도구 반복 제한 초과: 무한 실행하지 않습니다.")


## 관찰
모델 출력에 `function_call`이 있고 Python이 함수를 실행한 뒤 `call_id`를 연결해 결과를 되돌려줍니다. 함수 설명을 바꾸면 선택이 달라질 수 있습니다.

## 연습
① 날씨만 질문하기 ② 지원하지 않는 도시를 질문하기 ③ `tool_choice="none"`을 지정하면 실제 도구가 실행되는지 비교해 보세요.

<details><summary>예시 해설</summary>① 일반적으로 날씨 함수만 요청합니다. ② 추가 확인 또는 지원 도시 오류가 필요합니다. ③ 도구 요청이 금지되므로 실시간 날씨를 안다고 말하면 안 됩니다. 다음 노트북은 이 수동 반복을 Framework로 바꿉니다.</details>